[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gretelai/gretel-blueprints/blob/main/sdk_blueprints/Gretel_Data_Fidelity_Blueprint.ipynb)

<br>

<center><a href=https://gretel.ai/><img src="https://gretel-public-website.s3.us-west-2.amazonaws.com/assets/brand/gretel_brand_wordmark.svg" alt="Gretel" width="350"/></a></center>

<br>

## Welcome to the Gretel Data Fidelity Blueprint!

In this notebook, we show case M1 Requirements of **data fidelity**—a set of rules ensuring your synthetic data **follows specific constraints** as expected.

The **TabFT model** works well out-of-the-box, but sometimes, your data needs to be **extra precise**. That’s where **data fidelity** comes in! 🎯


## 🔍 **The Four Golden Rules of Data Fidelity (m1 requirements)**
1️⃣ **Rule #1:** Values in **Column A** should be embedded somewhere in **Column B**. For example, Email should include first and last name.

2️⃣ **Rule #2:** **Column A > Column B**. For example, admit date should always be <= discharge date.  

3️⃣ **Rule #3:** **DateTime minimum < Column A (DateTime) < DateTime maximum** Enforce minimum & maximum constraints on datetime values. For example, transactions should occur between certain dates like 1/1/2024 and 10/25/2024.  

4️⃣ **Rule #4:** Generate **highly unique values**. such as 'ID' column. 


## **🔧 Step 1: Install Necessary Libraries**

In [ ]:
!uv pip install git+https://github.com/gretelai/gretel-python-client@main pandas

import pandas as pd
from gretel_client.navigator_client import Gretel

## **🛜  Step 2: Configure your Gretel session:**

- Each `Gretel` instance is bound to a single [Gretel project](https://docs.gretel.ai/guides/gretel-fundamentals/projects).  
- You can retrieve your API key [here](https://console.gretel.ai/users/me/key).

In [ ]:
gretel = Gretel(
    api_key="prompt",
    validate = True,

)

## 🗂 **Step 3: Load and undertsand the Dataset**
Let's load our dataset containing real-world car accident data.

In [ ]:
dataset = pd.read_csv("https://gretel-datasets.s3.us-west-2.amazonaws.com/car_accident_5k.csv") # cited papers: [Moosavi, Sobhan, Mohammad Hossein Samavatian, Srinivasan Parthasarathy, and Rajiv Ramnath. “A Countrywide Traffic Accident Dataset.”, 2019. & Moosavi, Sobhan, Mohammad Hossein Samavatian, Srinivasan Parthasarathy, Radu Teodorescu, and Rajiv Ramnath. "Accident Risk Prediction based on Heterogeneous Sparse Data: New Dataset and Insights." In proceedings of the 27th ACM SIGSPATIAL International Conference on Advances in Geographic Information Systems, ACM, 2019.]
dataset.head()

## Understanding the Constraints in the Car Accident Data:
Following constraints exist in our real-world dataset, and we want to make sure they are replicated in the synthetic data:

- Rule #1: The last digit in `ID` corresponds to the `Severity` value of the accident.
- Rule #2: `End_Time` > `Start_Time` (Accident's end time must always be after start time).
- Rule #3: `Birth_Date` must be between `1931-01-03` and `1997-12-27` to ensure realistic personas in the car accident.
- Rule #4: `ID` values follow a strict pattern and are highly unique values. They start with `A-` and have the numeric values to be in range of (3100001, 7999994)

## 🛠 **Step 4: Generate Synthetic Data with Constraints**

Now, let's spin up a TabFT job that follows these rules.

In [ ]:
workflow_run = gretel.safe_synthetic_dataset\
.from_data_source(dataset)\
.synthesize(
    "tabular_ft",

    config={

        "train": {
            "data_config": {
                "columns": [ # Any DateTime columns should be specified with the type datetime and desired format.
                    {
                        "name": "Start_Time",
                        "type": "datetime",
                        "format": "%Y-%m-%d %H:%M:%S",
                    },
                    {
                        "name": "End_Time",
                        "type": "datetime",
                        "format": "%Y-%m-%d %H:%M:%S",
                    },
                    {
                        "name": "Driver_Birth_Date",
                        "type": "datetime",
                        "format": "%Y-%m-%d",
                    },
                ],

                "actions": [

                    {   "type": "date_constraint", # Make sure the start time is before the end time. Rule #2
                        "colA": "Start_Time",
                        "colB": "End_Time",
                        "operator": "lt"
                    },

                    {  "type": "expression_drop", # Make sure the driver's birth date is in the same range as the training data. Rule #3
                        "conditions": ["(row.Driver_Birth_Date | date_parse) > ('1997-12-27' | date_parse) or ((row.Driver_Birth_Date | date_parse) < ('1931-01-03'| date_parse))"]
                    },

                    { "type": "replace_datasource", # Rule #1 and #4.
                        "col": "ID",
                        "data_source" : {
                            "type": "expression",
                            "expression": '"A-" + (random.randint(310000,799999) | string)+ (row.Severity | string )',
                        } ,

                    },
                ],
            }
        },
        "generate": {
            "num_records": 1000,
        },
    },
)\
.create()


workflow_run.wait_until_done()



## **📊 Step 5: View the report of the dataset**

In [ ]:
workflow_run.report.table

## **🔎 Step 6: Validate Business Rules**
Now let's ensure our **synthetic dataset obeys the defined constraints**.

In [ ]:
# You can check the generated datasets using either of the following methods:
#1: Using the completed job from this notebook:
generated_df = workflow_run.dataset.df
#2: get the workflow run ID from console: (This is usually helpful if the notebook is interupted and you don't have access to `workflow_run`)
# workflow = gretel.workflows.get_workflow_run("your_worflow_run_ID")


# Validate Rules
Each rule should be retained and evaluated as True:

In [ ]:

# Rule #1
(generated_df['ID'].apply(lambda x: int(str(x)[-1])) == generated_df.Severity).all()

In [ ]:

# Rule #2:
(generated_df.Start_Time < generated_df.End_Time).all()



In [ ]:
# Rule #3:
generated_df['Driver_Birth_Date'] = pd.to_datetime(generated_df['Driver_Birth_Date'], errors='coerce')
((generated_df.Driver_Birth_Date <= pd.to_datetime("1997-12-27")) & (generated_df.Driver_Birth_Date >= pd.to_datetime("1931-01-03"))).all()


In [ ]:

#Rule #4:
# Part 1: Highly unique values (>95%) of ID:
(generated_df["ID"].nunique()/len(generated_df)) > 0.95

In [ ]:
#Rule #4:
# Part 2: IDs following specific pattern starting with "A-":
generated_df["ID"].apply(lambda x:x.startswith("A-")).all()

# Numeric section of the ID should be within a certain range:
min_range = 3100001
max_range = 7999994
generated_df["ID"].apply(lambda x:int(x.split("-")[1])).between(min_range, max_range).all()
